# CarSathi AI — 03 Data Cleaning & ML Dataset Preparation

Data cleaning, numeric extraction, torque conversion, duplicate handling, missing-value strategy, validation, and creation of the ML-ready dataset.


## Prerequisites / Setup


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path(
    r"C:\Users\lenovo\Document_local\Data Science NareshIT Course"
    r"\Projects For GITHUB\CarSathi-AI\data\raw\Car details v3.csv"
)

df = pd.read_csv(RAW_PATH)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

print("Dataset loaded successfully.")
print("Shape:", df.shape)


# $Step$ $4:$ $Data$ $Cleaning$

### $Duplicates$

In [ ]:
print("Before:", df.shape)

duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

print("After:", df.shape)

In [ ]:
df["mileage"] = df["mileage"].str.extract(r"([\d.]+)").astype(float)

In [ ]:
print(df["mileage"].head(10))
print(df["mileage"].dtype)

### $Clean$ $engine$

In [ ]:
df["engine"] = df["engine"].str.extract(r"([\d.]+)").astype(float)

In [ ]:
print(df["engine"].head(10))
print(df["engine"].dtype)

### $Clean$ $max_power$

In [ ]:
df["max_power"] = df["max_power"].str.extract(r"([\d.]+)").astype(float)

In [ ]:
print(df["max_power"].head(10))
print(df["max_power"].dtype)

In [ ]:
print(df[["mileage", "engine", "max_power"]].dtypes)

print(df[["mileage", "engine", "max_power"]].head(10))

# $Clean$ $torque$

In [ ]:
print(df["torque"].dropna().head(30).to_string(index=False))

In [ ]:
print("Unique torque values:", df["torque"].nunique())

In [ ]:
df.isnull().sum()

In [ ]:
print("Shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

In [ ]:
print(df["torque"].dropna().tail(30).to_string(index=False))

In [ ]:
print("Unique torque values:", df["torque"].nunique())

### $Create$ $a$ $clean$ $torque$ $column$

In [ ]:
import re

def clean_torque(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    # Case 1: Torque already in Nm
    nm_match = re.search(r'([\d.]+)\s*[Nn][Mm]', value)

    if nm_match:
        return float(nm_match.group(1))

    # Case 2: Torque in kgm
    kgm_match = re.search(r'([\d.]+)\s*(?:kgm|kg-m)', value, re.IGNORECASE)

    if kgm_match:
        kgm_value = float(kgm_match.group(1))
        return kgm_value * 9.80665

    # Case 3: Format such as 13.5@ 4,800(kgm@ rpm)
    kgm_format = re.search(r'([\d.]+)\s*@.*\(kgm@', value, re.IGNORECASE)

    if kgm_format:
        kgm_value = float(kgm_format.group(1))
        return kgm_value * 9.80665

    return None

In [ ]:
df["torque_nm"] = df["torque"].apply(clean_torque)

In [ ]:
print(df[["torque", "torque_nm"]].head(30).to_string(index=False))

In [ ]:
### Check whether anything failed

In [ ]:
print("Original missing torque:", df["torque"].isna().sum())

print("Cleaned missing torque:", df["torque_nm"].isna().sum())

In [ ]:
print(df["torque_nm"].describe())

In [ ]:
# Check the converted kgm values

In [ ]:
df[df["torque"].astype(str).str.contains("kgm", case=False, na=False)][
    ["torque", "torque_nm"]
].head(20)

In [ ]:
# First inspect the missing rows

In [ ]:
missing_cols = ["mileage", "engine", "max_power", "torque_nm", "seats"]

print(df[missing_cols].isnull().sum())

In [ ]:
df[df[missing_cols].isnull().any(axis=1)][
    ["name", "year", "selling_price", "fuel", "mileage",
     "engine", "max_power", "torque_nm", "seats"]
].head(20)

In [ ]:
# Check how many complete rows we have

In [ ]:
print("Total rows:", len(df))

print(
    "Rows with any missing value:",
    df[missing_cols].isnull().any(axis=1).sum()
)

print(
    "Rows with no missing value:",
    df[missing_cols].notnull().all(axis=1).sum()
)

In [ ]:
# Don't fill missing values blindly

In [ ]:
df["engine"].fillna(df["engine"].mean())

In [ ]:
# Seats

In [ ]:
print(df["seats"].value_counts(dropna=False).sort_index())

In [ ]:
print(df["seats"].describe())

In [ ]:
# Final Data Quality Check

In [ ]:
print("Shape:", df.shape)

print("\nMissing:")
print(df.isnull().sum())

print("\nDuplicates:")
print(df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

In [ ]:
failed_torque = df[
    df["torque"].notna() &
    df["torque_nm"].isna()
]

print("Torque conversion failures:", len(failed_torque))

print(
    failed_torque[["torque"]]
    .drop_duplicates()
    .to_string(index=False)
)

In [ ]:
import re

def clean_torque(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    # 1. Explicit Nm formats
    nm_match = re.search(r'([\d.]+)\s*[Nn][Mm]', value)
    if nm_match:
        return float(nm_match.group(1))

    # 2. Explicit kgm formats
    kgm_match = re.search(r'([\d.]+)\s*(?:kgm|kg-m)', value, re.IGNORECASE)
    if kgm_match:
        return float(kgm_match.group(1)) * 9.80665

    # 3. Format: 13.5@ 4,800(kgm@ rpm)
    kgm_format = re.search(
        r'([\d.]+)\s*@.*\(kgm@',
        value,
        re.IGNORECASE
    )
    if kgm_format:
        return float(kgm_format.group(1)) * 9.80665

    # 4. Format: 48@ 3,000+/-500(NM@ rpm)
    nm_format = re.search(
        r'([\d.]+)\s*@.*\(NM@',
        value,
        re.IGNORECASE
    )
    if nm_format:
        return float(nm_format.group(1))

    # 5. Generic numeric torque formats:
    # 250@ 1250-5000rpm
    # 510@ 1600-2400
    # 110(11.2)@ 4800
    # 210 / 1900
    generic = re.match(r'\s*([\d.]+)', value)

    if generic:
        return float(generic.group(1))

    return None

In [ ]:
df["torque_nm"] = df["torque"].apply(clean_torque)

In [ ]:
failed_torque = df[
    df["torque"].notna() &
    df["torque_nm"].isna()
]

print("Torque conversion failures:", len(failed_torque))

In [ ]:
print(
    df[df["torque"].isin([
        "250@ 1250-5000rpm",
        "48@ 3,000+/-500(NM@ rpm)",
        "510@ 1600-2400",
        "110(11.2)@ 4800",
        "210 / 1900"
    ])][["torque", "torque_nm"]].to_string(index=False)
)

In [ ]:
print("Original torque missing:", df["torque"].isna().sum())
print("Clean torque missing:", df["torque_nm"].isna().sum())

In [ ]:
# Missing Values

In [ ]:
missing_cols = [
    "mileage",
    "engine",
    "max_power",
    "torque_nm",
    "seats"
]

print(df[missing_cols].isnull().sum())

In [ ]:
missing_cols = [
    "mileage",
    "engine",
    "max_power",
    "torque_nm",
    "seats"
]

missing_pattern = df[missing_cols].isnull()

print(missing_pattern.sum(axis=1).value_counts().sort_index())

In [ ]:
df[df[missing_cols].isnull().any(axis=1)][
    [
        "name",
        "year",
        "selling_price",
        "km_driven",
        "fuel",
        "transmission",
        "mileage",
        "engine",
        "max_power",
        "torque_nm",
        "seats"
    ]
].head(30)

In [ ]:
missing_report = pd.DataFrame({
    "Missing Count": df[missing_cols].isnull().sum(),
    "Missing %": (
        df[missing_cols].isnull().mean() * 100
    ).round(2)
})

print(missing_report)

In [ ]:
df[df["engine"].isnull()][
    ["name", "fuel", "selling_price", "engine"]
].head(20)

In [ ]:
df[df["mileage"].isnull()][
    ["name", "fuel", "selling_price", "mileage"]
].head(20)

In [ ]:
print(df["seats"].value_counts(dropna=False).sort_index())

In [ ]:
print(df["seats"].describe())

In [ ]:
df[df["seats"] == 14][
    ["name", "year", "selling_price", "fuel",
     "mileage", "engine", "max_power", "torque_nm", "seats"]
]

In [ ]:
df[df["seats"].isna()][
    ["name", "year", "selling_price", "fuel",
     "mileage", "engine", "max_power", "torque_nm", "seats"]
].head(30)

In [ ]:
print("Current shape:", df.shape)

raw_df = pd.read_csv(r"C:\Users\lenovo\Document_local\Data Science NareshIT Course\Projects For GITHUB\CarSathi-AI\data\raw\Car details v3.csv")

print("Raw shape:", raw_df.shape)

In [ ]:
current_names = set(df["name"])

removed_rows = raw_df[~raw_df["name"].isin(current_names)]

print("Rows potentially removed:", len(removed_rows))

In [ ]:
key_cols = ["name", "year", "selling_price", "km_driven"]

raw_keys = raw_df[key_cols].astype(str).agg("|".join, axis=1)
current_keys = df[key_cols].astype(str).agg("|".join, axis=1)

removed_rows = raw_df[~raw_keys.isin(set(current_keys))]

print("Rows removed:", len(removed_rows))

In [ ]:
print(removed_rows.head(20).to_string(index=False))

In [ ]:
# Compare raw dataset with current dataframe

key_cols = ["name", "year", "selling_price", "km_driven"]

raw_keys = raw_df[key_cols].astype(str).agg("|".join, axis=1)
current_keys = df[key_cols].astype(str).agg("|".join, axis=1)

removed_rows = raw_df[
    ~raw_keys.isin(set(current_keys))
]

print("Rows removed:", len(removed_rows))

print("\nFirst 20 removed rows:")
print(removed_rows.head(20).to_string(index=False))

In [ ]:
key_cols = ["name", "year", "selling_price", "km_driven"]

raw_counts = (
    raw_df.groupby(key_cols)
    .size()
    .reset_index(name="raw_count")
)

current_counts = (
    df.groupby(key_cols)
    .size()
    .reset_index(name="current_count")
)

comparison = raw_counts.merge(
    current_counts,
    on=key_cols,
    how="left"
)

comparison["current_count"] = comparison["current_count"].fillna(0)

comparison["rows_removed"] = (
    comparison["raw_count"] - comparison["current_count"]
)

print(
    comparison[comparison["rows_removed"] > 0]
    .sort_values("rows_removed", ascending=False)
    .head(20)
)

In [ ]:
print(
    "Total rows removed according to frequency comparison:",
    comparison["rows_removed"].sum()
)

In [ ]:
print("Raw rows:", len(raw_df))
print("Current rows:", len(df))
print("Difference:", len(raw_df) - len(df))

In [ ]:
key_cols = ["name", "year", "selling_price", "km_driven"]

raw_counts = (
    raw_df.groupby(key_cols)
    .size()
    .reset_index(name="raw_count")
)

current_counts = (
    df.groupby(key_cols)
    .size()
    .reset_index(name="current_count")
)

comparison = raw_counts.merge(
    current_counts,
    on=key_cols,
    how="left"
)

comparison["current_count"] = comparison["current_count"].fillna(0)

comparison["rows_removed"] = (
    comparison["raw_count"] - comparison["current_count"]
)

print(
    "Total rows removed:",
    comparison["rows_removed"].sum()
)

In [ ]:
print("Raw rows:", len(raw_df))
print("Current rows:", len(df))
print("Difference:", len(raw_df) - len(df))

In [ ]:
print(
    comparison[comparison["rows_removed"] > 0]
    .sort_values("rows_removed", ascending=False)
    .head(20)
    .to_string(index=False)
)

In [ ]:
print("Raw exact duplicates:", raw_df.duplicated().sum())

In [ ]:
print(
    raw_df[
        raw_df.duplicated(keep=False)
    ].head(20).to_string(index=False)
)

In [ ]:
print(
    "Current exact duplicates:",
    df.duplicated().sum()
)

# $Create$ $fresh$ $dataframe$

In [ ]:
cars = raw_df.copy()

print("Raw shape:", cars.shape)

# $Remove$ $exact$ $duplicates$

In [ ]:
cars = cars.drop_duplicates().reset_index(drop=True)

print("After duplicate removal:", cars.shape)

In [ ]:
print("Duplicates:", cars.duplicated().sum())

# $Clean$ $mileage$

In [ ]:
cars["mileage"] = (
    cars["mileage"]
    .str.extract(r"([\d.]+)", expand=False)
    .astype(float)
)

In [ ]:
print(cars["mileage"].head(10))
print(cars["mileage"].dtype)

# $Clean$ $engine$

In [ ]:
cars["engine"] = (
    cars["engine"]
    .str.extract(r"([\d.]+)", expand=False)
    .astype(float)
)

In [ ]:
print(cars["engine"].head(10))
print(cars["engine"].dtype)

# $Clean$ $maxpower$

In [ ]:
cars["max_power"] = (
    cars["max_power"]
    .str.extract(r"([\d.]+)", expand=False)
    .astype(float)
)

In [ ]:
print(cars["max_power"].head(10))
print(cars["max_power"].dtype)

# $Clean$ $Torque$

In [ ]:
import re

def clean_torque(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    # Nm
    nm_match = re.search(r'([\d.]+)\s*[Nn][Mm]', value)
    if nm_match:
        return float(nm_match.group(1))

    # kgm
    kgm_match = re.search(
        r'([\d.]+)\s*(?:kgm|kg-m)',
        value,
        re.IGNORECASE
    )

    if kgm_match:
        return float(kgm_match.group(1)) * 9.80665

    # Example: 13.5@ 4,800(kgm@ rpm)
    kgm_format = re.search(
        r'([\d.]+)\s*@.*\(kgm@',
        value,
        re.IGNORECASE
    )

    if kgm_format:
        return float(kgm_format.group(1)) * 9.80665

    # Example: 48@ 3,000+/-500(NM@ rpm)
    nm_format = re.search(
        r'([\d.]+)\s*@.*\(NM@',
        value,
        re.IGNORECASE
    )

    if nm_format:
        return float(nm_format.group(1))

    # Generic formats
    generic = re.match(r'\s*([\d.]+)', value)

    if generic:
        return float(generic.group(1))

    return None

In [ ]:
cars["torque_nm"] = cars["torque"].apply(clean_torque)

# $Check$ $all$ $numerical$ $columns$

In [ ]:
numeric_cols = [
    "mileage",
    "engine",
    "max_power",
    "torque_nm",
    "seats"
]

print(cars[numeric_cols].dtypes)

# $Check$ $missing$ $values$

In [ ]:
print(cars.isnull().sum())

# $Validate$ $the$ $cleaned$ $dataframe$

In [ ]:
print("Shape:", cars.shape)

print("\nDuplicates:", cars.duplicated().sum())

print("\nMissing values:")
print(cars.isnull().sum())

print("\nData types:")
print(cars.dtypes)

# $Missing-Value$ $Strategy$

# 1️⃣ $Check$ $missing$ $combinations$

In [ ]:
missing_cols = [
    "mileage",
    "engine",
    "max_power",
    "torque_nm",
    "seats"
]

missing_pattern = cars[missing_cols].isnull().sum(axis=1)

print(
    missing_pattern.value_counts()
    .sort_index()
)

# 2️⃣ $See$ $how$ $many$ $rows$ $have$ $all$ $specifications$ $missing$

In [ ]:
all_missing = cars[
    cars[missing_cols].isnull().all(axis=1)
]

print("All specifications missing:", len(all_missing))

print(
    all_missing[
        ["name", "year", "selling_price", "km_driven", "fuel"]
    ].head(30).to_string(index=False)
)

# 3️⃣ $Check$ $partial$ $missing$ $records$

In [ ]:
partial_missing = cars[
    cars[missing_cols].isnull().any(axis=1) &
    ~cars[missing_cols].isnull().all(axis=1)
]

print("Partial missing:", len(partial_missing))

print(
    partial_missing[
        [
            "name",
            "year",
            "selling_price",
            "fuel",
            "mileage",
            "engine",
            "max_power",
            "torque_nm",
            "seats"
        ]
    ].head(30).to_string(index=False)
)

# $Create$ $the$ $ML$ $dataset$

In [ ]:
cars_ml = cars.copy()

In [ ]:
cars_ml = cars_ml[
    ~cars_ml[missing_cols].isnull().all(axis=1)
].copy()

cars_ml = cars_ml.reset_index(drop=True)

In [ ]:
print("Original cleaned dataset:", cars.shape)
print("ML dataset:", cars_ml.shape)

# $Convert$ $zero$ $power$ $to$ $missing$

In [ ]:
import numpy as np

In [ ]:
cars_ml["max_power"] = cars_ml["max_power"].replace(0, np.nan)

In [ ]:
print(
    cars_ml[cars_ml["max_power"].isna()][
        ["name", "max_power"]
    ].to_string(index=False)
)

In [ ]:
missing_ml = cars_ml[
    cars_ml[missing_cols].isnull().any(axis=1)
]

print(
    missing_ml[
        ["name", "year", "fuel", "mileage",
         "engine", "max_power", "torque_nm", "seats"]
    ].to_string(index=False)
)

In [ ]:
for car_name in missing_ml["name"].unique():
    print("\n==============================")
    print(car_name)
    print("==============================")
    
    print(
        cars_ml[
            cars_ml["name"] == car_name
        ][
            ["name", "year", "fuel", "mileage",
             "engine", "max_power", "torque_nm", "seats"]
        ].head(10).to_string(index=False)
    )

In [ ]:
cars_ml = cars.copy()

cars_ml = cars_ml[
    ~cars_ml[missing_cols].isnull().all(axis=1)
].copy()

cars_ml = cars_ml.reset_index(drop=True)

cars_ml["max_power"] = cars_ml["max_power"].replace(0, np.nan)

print("Master cleaned dataset:", cars.shape)
print("ML dataset:", cars_ml.shape)

In [ ]:
missing_ml = cars_ml[
    cars_ml[missing_cols].isnull().any(axis=1)
]

print(
    missing_ml[
        ["name", "year", "fuel", "mileage",
         "engine", "max_power", "torque_nm", "seats"]
    ].to_string(index=False)
)

In [ ]:
impute_cols = [
    "mileage",
    "engine",
    "max_power",
    "torque_nm",
    "seats"
]

for col in impute_cols:
    cars_ml[col] = cars_ml[col].fillna(
        cars_ml.groupby(["name", "fuel"])[col]
        .transform("median")
    )

In [ ]:
print(cars_ml[impute_cols].isnull().sum())

In [ ]:
remaining_missing = cars_ml[
    cars_ml[impute_cols].isnull().any(axis=1)
]

print(
    remaining_missing[
        ["name", "year", "fuel", "mileage",
         "engine", "max_power", "torque_nm", "seats"]
    ].to_string(index=False)
)

In [ ]:
print(
    cars_ml[
        cars_ml["name"].str.contains(
            "Maruti Omni",
            case=False,
            na=False
        )
    ][
        ["name", "year", "fuel", "mileage",
         "engine", "max_power", "torque_nm", "seats"]
    ].to_string(index=False)
)

In [ ]:
print(
    cars_ml[
        cars_ml["fuel"].str.upper().eq("CNG")
    ][
        ["name", "year", "fuel", "mileage",
         "engine", "max_power", "torque_nm", "seats"]
    ].to_string(index=False)
)

In [ ]:
cng_796 = cars_ml[
    (cars_ml["fuel"] == "CNG") &
    (cars_ml["engine"] == 796)
]

print(cng_796[["name", "max_power", "torque_nm"]])


In [ ]:
print("Median max power:", cng_796["max_power"].median())
print("Median torque:", cng_796["torque_nm"].median())

In [ ]:
# Fill the remaining missing values for Maruti Omni CNG
omni_mask = (
    (cars_ml["name"] == "Maruti Omni CNG") &
    (cars_ml["fuel"] == "CNG")
)

cars_ml.loc[omni_mask, "max_power"] = 40.3
cars_ml.loc[omni_mask, "torque_nm"] = 62.0

In [ ]:
print(
    cars_ml.loc[
        omni_mask,
        ["name", "year", "fuel", "mileage",
         "engine", "max_power", "torque_nm", "seats"]
    ]
)

In [ ]:
print("Final missing values:")
print(cars_ml.isnull().sum())

In [ ]:
print(
    "Total missing cells:",
    cars_ml.isnull().sum().sum()
)

In [ ]:
print("Final ML dataset shape:", cars_ml.shape)

In [ ]:
print(cars_ml[
    ["selling_price",
     "km_driven",
     "mileage",
     "engine",
     "max_power",
     "torque_nm",
     "seats"]
].describe())

In [ ]:
cars_ml = cars_ml.drop(columns=["torque"])

print("Final shape:", cars_ml.shape)
print("\nMissing values:")
print(cars_ml.isnull().sum())

In [ ]:
print("Mileage = 0:")
print(
    cars_ml[cars_ml["mileage"] == 0][
        ["name", "year", "selling_price",
         "fuel", "mileage", "engine",
         "max_power", "torque_nm", "seats"]
    ].to_string(index=False)
)

In [ ]:
print("\nVery high torque:")
print(
    cars_ml[cars_ml["torque_nm"] > 500][
        ["name", "year", "selling_price",
         "fuel", "engine", "max_power",
         "torque_nm", "seats"]
    ].sort_values(
        "torque_nm",
        ascending=False
    ).to_string(index=False)
)

In [ ]:
print("\nVery high power:")
print(
    cars_ml[cars_ml["max_power"] > 250][
        ["name", "year", "selling_price",
         "fuel", "engine", "max_power",
         "torque_nm", "seats"]
    ].sort_values(
        "max_power",
        ascending=False
    ).to_string(index=False)
)

In [ ]:
cars_ml[cars_ml["torque_nm"] > 700][
    ["name", "year", "fuel", "engine", "max_power", "torque_nm", "seats"]
].sort_values("torque_nm", ascending=False)

## Notebook Complete


In [ ]:
# Save the cleaned ML-ready dataset for the next notebooks.
from pathlib import Path

PROCESSED_DIR = Path(
    r"C:\Users\lenovo\Document_local\Data Science NareshIT Course"
    r"\Projects For GITHUB\CarSathi-AI\data\processed"
)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DIR / "cars_ml.csv"
cars_ml.to_csv(output_path, index=False)

print(f"Saved cleaned ML dataset to: {output_path}")
print("Final shape:", cars_ml.shape)
